In [2]:
import requests
import json
import pandas as pd
import numpy as np
from django.contrib.admin import display

'''
url padrao
https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados?formato=json&dataInicial={dd/MM/yyyy}&dataFinal={dd/MM/yyyy}
https ://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/[codigo_recurso]?$format=json&[Outros Parâmetros]
'''


class Coletor:
    def __init__(self):
        self.dataInicial = None
        self.dataFinal = None

    def coletor(self, dataInicial=None, dataFinal=None):

        #Data
        print("Formato DIA/MES/ANO\nex: 31/01/2021")
        dataInicial = dataInicial if dataInicial is not None else input("Digite a data inicial: ")
        dataFinal = dataFinal if dataFinal is not None else input("Digite a data final: ")

        #Data Cambio
        dataInicial_ptax = dataInicial.replace("/","-")
        dataFinal_ptax = dataFinal.replace("/","-")

        #URL's
        url_selic = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.11/dados"
        url_ipca = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados"
        url_cambio = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial='{dataInicial_ptax}',dataFinalCotacao='{dataFinal_ptax}')?$format=json"

        parametros = {
            "formato": "json",
            "dataInicial": dataInicial,
            "dataFinal": dataFinal
        }

        # GET SELIC
        r_selic = requests.get(url_selic, params=parametros)
        dados_selic = r_selic.json()

        #Tratamento de Dados SELIC
        df_selic = pd.DataFrame(dados_selic)  # A partir daqui vira um DataFrame do pandas.
        df_selic["valor"] = df_selic["valor"].str.replace(",", ".")  # A API recebe o valor apenas com virgula.
        df_selic["valor"] = pd.to_numeric(df_selic["valor"])  # valores se tornam float
        df_selic["data"] = pd.to_datetime(df_selic["data"],format="%d/%m/%Y")  # converter para o formato datetime do pandas
        df_selic.to_json("dados_selic.json", orient="records",date_format="iso")  # salva o arquivo em json e aplica o formato iso na coluna "data"

        # GET IPCA
        r_ipca = requests.get(url_ipca, params=parametros)
        dados_ipca = r_ipca.json()

        #Tratamento de Dados IPCA
        df_ipca = pd.DataFrame(dados_ipca)
        df_ipca["valor"] = df_ipca["valor"].str.replace(",", ".")
        df_ipca["valor"] = pd.to_numeric(df_ipca["valor"])
        df_ipca["data"] = pd.to_datetime(df_ipca["data"], format="%d/%m/%Y")
        df_ipca.to_json("dados_ipca.json", orient="records",date_format="iso")

        # Cambio
        r_cambio = requests.get(url_cambio)
        dados_cambio = r_cambio.json()
        # print(dados_cambio)
        df_cambio = pd.DataFrame(dados_cambio["value"])
        print(df_cambio)
        df_cambio.to_json("../dados/dados_cambio.json",orient="index", date_format="iso", indent=4)
        display(df_cambio)
        return dataInicial, dataFinal



In [3]:
coletor = Coletor()
coletor.coletor()

Formato DIA/MES/ANO
ex: 31/01/2021
   cotacaoCompra  cotacaoVenda          dataHoraCotacao
0         4.0207        4.0213  2020-01-02 13:11:10.762
1         4.0516        4.0522  2020-01-03 13:06:22.606
2         4.0548        4.0554  2020-01-06 13:03:22.271
3         4.0835        4.0841  2020-01-07 13:06:14.601
4         4.0666        4.0672  2020-01-08 13:03:56.075
5         4.0738        4.0744  2020-01-09 13:03:52.663
6         4.0739        4.0745  2020-01-10 13:10:19.927


('01/01/2020', '01/12/2020')